### Raster Model
This notebook contains the workflow for downscaling national capital and labor data. First, it uses the RF model to predict intensities at the pixel scale. Then it combines intensities with rasterized production data to get initial estimates of capital and labor in each pixel. Then it runs a re-scaling algorithim to ensure estimates align with sub-national (where available) and national totals. The output is a set of raster datasets contained estimates of capital (million USD) and labor (jobs) in each pixel. 

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import joblib
import rasterio
import gc

In [2]:
##### SET-UP

### Set directories 
# Get the current working directory
cd = Path.cwd().parent.parent 

# Set directories for saving
RESULTS_DIR = Path(f"{cd}/Results/Raster_model")

### Import data 
# RF models 
capital_rf = joblib.load(f'{cd}/Results/RF_models_final/capital_rf_final/model.joblib')
capital_qrf = joblib.load(f'{cd}/Results/RF_models_final/capital_qrf_final/model.joblib')

labor_rf = joblib.load(f'{cd}/Results/RF_models_final/labor_rf_final/model.joblib')
labor_qrf = joblib.load(f'{cd}/Results/RF_models_final/labor_qrf_final/model.joblib')

# Predictor data (stacked rasters stored in df)
predictors = pd.read_parquet(f'{cd}/Data/Clean/Training_data/raster_predictor_matrix_final.parquet')

# Production data (raster)
production_path = f'{cd}/Data/Clean/Production/total_production_tonnes_2020.tif'

# Sub-national data (for re-scaling)
capital_sub = pd.read_csv(f'{cd}/Data/Clean/Capital_stock/subnational_capital_stock_final.csv')
capital_sub_crosswalk = pd.read_parquet(f'{cd}/Data/Clean/Geographies/pixel_sub_capital_crosswalk.parquet')

labor_sub = pd.read_csv(f'{cd}/Data/Clean/Labor/subnational_labor_final.csv')
labor_sub_crosswalk = pd.read_parquet(f'{cd}/Data/Clean/Geographies/pixel_sub_labor_crosswalk.parquet')

# National data (for re-scaling)
capital_national = pd.read_csv(f'{cd}/Data/Clean/Capital_stock/FAO_capital_stock_adjusted.csv')
labor_national = pd.read_csv(f'{cd}/Data/Clean/Labor/ILO_ag_labor_estimate_adjusted.csv')
country_crosswalk = pd.read_parquet(f'{cd}/Data/Clean/Geographies/pixel_country_crosswalk.parquet')

#### Part 1: Predict intensities

In [3]:
##### PREDICTOR COLUMNS

capital_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_oilcrops_share_base_100_production_USD',
       'rtv_log_pulses_share_base_100_production_USD',
       'rtv_log_roots_tubers_share_base_100_production_USD',
       'rtv_log_sugar_crops_share_base_100_production_USD',
       'rtv_log_ruminants_share_base_100_production_USD']

labor_cols = ['rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_ruminants_share_base_100_production_USD']

In [ ]:
##### PREDICT INTENSITIES 

# reduce parallelism to limit memory multiplication across worker processes
capital_qrf.n_jobs = 4
labor_qrf.n_jobs = 4
capital_rf.n_jobs = 4
labor_rf.n_jobs = 4

chunk_size = 50_000  # change as needed based on computer 
n = len(predictors)

# set output columns
out_cols = [
    'rtv_log_capital_intensity_USD_per_million_tonne',
    'rtv_log_labor_intensity_jobs_per_million_tonne',
    'rtv_log_capital_intensity_p10',
    'rtv_log_capital_intensity_p90',
    'rtv_log_labor_intensity_p10',
    'rtv_log_labor_intensity_p90',
]

for c in out_cols:
    predictors[c] = np.nan

# predict!
for start in range(0, n, chunk_size):
    end = min(start + chunk_size, n)
    idx = predictors.index[start:end]

    X_cap_chunk = predictors.loc[idx, capital_cols].astype('float32')
    X_lab_chunk = predictors.loc[idx, labor_cols].astype('float32')

    predictors.loc[idx, 'rtv_log_capital_intensity_USD_per_million_tonne'] = capital_rf.predict(X_cap_chunk)
    predictors.loc[idx, 'rtv_log_labor_intensity_jobs_per_million_tonne'] = labor_rf.predict(X_lab_chunk)

    cap_q = capital_qrf.predict(X_cap_chunk, quantiles=[0.1, 0.9])
    predictors.loc[idx, 'rtv_log_capital_intensity_p10'] = cap_q[:, 0]
    predictors.loc[idx, 'rtv_log_capital_intensity_p90'] = cap_q[:, 1]

    lab_q = labor_qrf.predict(X_lab_chunk, quantiles=[0.1, 0.9])
    predictors.loc[idx, 'rtv_log_labor_intensity_p10'] = lab_q[:, 0]
    predictors.loc[idx, 'rtv_log_labor_intensity_p90'] = lab_q[:, 1]

    del X_cap_chunk, X_lab_chunk, cap_q, lab_q
    gc.collect()

    print(f"done {end}/{n}")

# Filter 
cols_to_keep = ['x', 'y',
    'rtv_log_capital_intensity_USD_per_million_tonne',
    'rtv_log_labor_intensity_jobs_per_million_tonne',
    'rtv_log_capital_intensity_p10',
    'rtv_log_capital_intensity_p90',
    'rtv_log_labor_intensity_p10',
    'rtv_log_labor_intensity_p90',
]
raster_predictions = predictors[cols_to_keep]

In [5]:
##### CONVERT UNITS

# add back country intensities 
raster_predictions = raster_predictions.merge(predictors[['x', 'y', 'log_country_capital_intensity_USD_per_million_tonne', 'log_country_labor_intensity_jobs_per_million_tonne']], on=['x', 'y'], how='outer')

# convert pixel intensities from relative to absolute 
raster_predictions['capital_intensity_USD_per_million_tonne'] = np.expm1(raster_predictions['rtv_log_capital_intensity_USD_per_million_tonne'] + raster_predictions['log_country_capital_intensity_USD_per_million_tonne'])
raster_predictions['capital_intensity_p10'] = np.expm1(raster_predictions['rtv_log_capital_intensity_p10'] + raster_predictions['log_country_capital_intensity_USD_per_million_tonne'])
raster_predictions['capital_intensity_p90'] = np.expm1(raster_predictions['rtv_log_capital_intensity_p90'] + raster_predictions['log_country_capital_intensity_USD_per_million_tonne'])

raster_predictions['labor_intensity_jobs_per_million_tonne'] = np.expm1(raster_predictions['rtv_log_labor_intensity_jobs_per_million_tonne'] + raster_predictions['log_country_labor_intensity_jobs_per_million_tonne'])
raster_predictions['labor_intensity_p10'] = np.expm1(raster_predictions['rtv_log_labor_intensity_p10'] + raster_predictions['log_country_labor_intensity_jobs_per_million_tonne'])
raster_predictions['labor_intensity_p90'] = np.expm1(raster_predictions['rtv_log_labor_intensity_p90'] + raster_predictions['log_country_labor_intensity_jobs_per_million_tonne'])

# save intermediate output
raster_predictions.to_parquet(RESULTS_DIR / "intermediate_outputs/intensity_predictions.parquet", index='False')

#### Part 2: Initial capital and labor estimates

In [6]:
##### PREP PRODUCTION DATA 

# convert production tiff into df with x/y coords 
# both prediction and production raster in EPSG:4326, 5 arcmin - so x/y match 
with rasterio.open(production_path) as src:
    arr = src.read(1).astype(np.float32)
    nodata = src.nodata
    if nodata is not None:
        arr[arr == nodata] = np.nan

    height, width = arr.shape
    transform = src.transform
    rows, cols = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
    xs, ys = rasterio.transform.xy(transform, rows.ravel(), cols.ravel())
    xs = np.array(xs)
    ys = np.array(ys)

production_df = pd.DataFrame({
    "x": xs,
    "y": ys,
    "total_production_tonnes_2020": arr.ravel()
})

In [8]:
##### CONVERT INTENSITIES TO CAPITAL AND LABOR

# Merge intensities and production
raster_predictions = raster_predictions.merge(production_df, on=['x', 'y'], how='outer')

# Capital and labor calculations 
raster_predictions['capital_USD'] = raster_predictions['capital_intensity_USD_per_million_tonne'] * (raster_predictions['total_production_tonnes_2020'] /  1e6)
raster_predictions['capital_USD_p10'] = raster_predictions['capital_intensity_p10'] * (raster_predictions['total_production_tonnes_2020'] /  1e6)
raster_predictions['capital_USD_p90'] = raster_predictions['capital_intensity_p90'] * (raster_predictions['total_production_tonnes_2020'] /  1e6)

raster_predictions['jobs'] = raster_predictions['labor_intensity_jobs_per_million_tonne'] * (raster_predictions['total_production_tonnes_2020'] /  1e6)
raster_predictions['jobs_p10'] = raster_predictions['labor_intensity_p10'] * (raster_predictions['total_production_tonnes_2020'] /  1e6)
raster_predictions['jobs_p90'] = raster_predictions['labor_intensity_p90'] * (raster_predictions['total_production_tonnes_2020'] /  1e6)

# Filter columns 
col_to_keep = ['x', 'y', 'total_production_tonnes_2020', 
               'capital_USD', 'capital_USD_p10', 'capital_USD_p90', 
               'jobs', 'jobs_p10', 'jobs_p90'
               ]
capital_labor_predictions = raster_predictions[col_to_keep]

# save intermediate output
capital_labor_predictions.to_parquet(RESULTS_DIR / "intermediate_outputs/capital_labor_predictions.parquet", index='False')

In [9]:
##### SAVE INTERMEDIATE RASTERS

# define function to convert to raster 
def save_column_as_raster(df, value_col, reference_raster_path, output_path):

    # use production raster as reference grid
    with rasterio.open(reference_raster_path) as ref:
        transform = ref.transform
        crs = ref.crs
        height = ref.height
        width = ref.width

    # pivot into 2D grid: rows = y, cols = x
    pivot = df.pivot(index='y', columns='x', values=value_col)

    # match raster row/col order: y descending (north to south), x ascending
    pivot = pivot.sort_index(ascending=False)
    pivot = pivot.reindex(columns=sorted(pivot.columns))

    arr = pivot.values.astype(np.float32)

    if arr.shape != (height, width):
        raise ValueError(
            f"Shape mismatch for {value_col}: got {arr.shape}, expected {(height, width)}"
        )

    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype='float32',
        crs=crs,
        transform=transform,
        nodata=np.nan
    ) as dst:
        dst.write(arr, 1)

# run for all columns 
output_cols = [
    'capital_USD',
    'capital_USD_p10',
    'capital_USD_p90',
    'jobs',
    'jobs_p10',
    'jobs_p90'
]

for col in output_cols:
    out_path = RESULTS_DIR / f"intermediate_outputs/pre_scaled_{col}.tif" 
    save_column_as_raster(raster_predictions, col, production_path, out_path)
    print(f"Saved {col} -> {out_path}")

Saved capital_USD -> /Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/intermediate_outputs/pre_scaled_capital_USD.tif
Saved capital_USD_p10 -> /Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/intermediate_outputs/pre_scaled_capital_USD_p10.tif
Saved capital_USD_p90 -> /Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/intermediate_outputs/pre_scaled_capital_USD_p90.tif
Saved jobs -> /Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/intermediate_outputs/pre_scaled_jobs.tif
Saved jobs_p10 -> /Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/intermediate_outputs/pre_scaled_jobs_p10.tif
Saved jobs_p90 -> /Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/intermediate_outputs/pre_scaled_jobs_p90.tif


#### Part 3: rescaling to sub-national and national totals

In [10]:
##### RE-SCALE TO SUB-NATIONAL (where it exists)

capital_labor_predictions = pd.read_parquet(RESULTS_DIR / "intermediate_outputs/capital_labor_predictions.parquet")

### Step 1: Sum total predicted capital and labor in each sub-national region
# Add cross-walk of which sub-national region each pixel is in 
capital_sub_crosswalk = capital_sub_crosswalk.rename(columns={'PROJ_ID': 'CAPITAL_PROJ_ID'})
labor_sub_crosswalk = labor_sub_crosswalk.rename(columns={'PROJ_ID': 'LABOR_PROJ_ID'})

capital_labor_predictions = capital_labor_predictions.merge(capital_sub_crosswalk, on=['x', 'y'], how='left')
capital_labor_predictions = capital_labor_predictions.merge(labor_sub_crosswalk, on=['x', 'y'], how='left')

# Get totals from initial predictions by sub-national region 
capital_sub_national_predictions = capital_labor_predictions[['CAPITAL_PROJ_ID', 'capital_USD', 'capital_USD_p10', 'capital_USD_p90']].groupby('CAPITAL_PROJ_ID').sum().reset_index()
labor_sub_national_predictions = capital_labor_predictions[['LABOR_PROJ_ID', 'jobs', 'jobs_p10', 'jobs_p90']].groupby('LABOR_PROJ_ID').sum().reset_index()


### Step 2: Get re-scaling factor for each sub-national region

# For capital, drop sub-national regions in countries where a proxy was used to aprx sub-national totals 
countries_to_drop = ['BRA', 'EGY', 'GHA', 'IND', 'TUR', 'ZAF', 'TZA', 'ARG']
capital_sub = capital_sub[~capital_sub["ISO3"].isin(countries_to_drop)]

# Enforce no duplicates
capital_sub = capital_sub[~capital_sub.duplicated(subset=['PROJ_ID'], keep=False)]
labor_sub = labor_sub[~labor_sub.duplicated(subset=['PROJ_ID'], keep=False)]

# Join predicted capital and labor to actual 
capital_sub = capital_sub.rename(columns={'ag_capital_stock_USD_nominal': 'sub_national_total_capital_USD'})
labor_sub = labor_sub.rename(columns={'ag_jobs': 'sub_national_total_jobs'})

capital_sub_national_predictions = capital_sub_national_predictions.merge(capital_sub[['PROJ_ID', 'sub_national_total_capital_USD']], left_on='CAPITAL_PROJ_ID', right_on='PROJ_ID', how='left')
labor_sub_national_predictions = labor_sub_national_predictions.merge(labor_sub[['PROJ_ID', 'sub_national_total_jobs']], left_on='LABOR_PROJ_ID', right_on='PROJ_ID', how='left')

# Calculate rescaling factors 
capital_sub_national_predictions['capital_sub_scale'] = capital_sub_national_predictions['sub_national_total_capital_USD'] / capital_sub_national_predictions['capital_USD'] 
capital_sub_national_predictions['capital_sub_scale_p10'] = capital_sub_national_predictions['sub_national_total_capital_USD'] / capital_sub_national_predictions['capital_USD_p10'] 
capital_sub_national_predictions['capital_sub_scale_p90'] = capital_sub_national_predictions['sub_national_total_capital_USD'] / capital_sub_national_predictions['capital_USD_p90'] 

labor_sub_national_predictions['labor_sub_scale'] = labor_sub_national_predictions['sub_national_total_jobs'] / labor_sub_national_predictions['jobs'] 
labor_sub_national_predictions['labor_sub_scale_p10'] = labor_sub_national_predictions['sub_national_total_jobs'] / labor_sub_national_predictions['jobs_p10'] 
labor_sub_national_predictions['labor_sub_scale_p90'] = labor_sub_national_predictions['sub_national_total_jobs'] / labor_sub_national_predictions['jobs_p90'] 

# Merge back to data 
capital_sub_national_predictions = capital_sub_national_predictions.drop(columns=['capital_USD', 'capital_USD_p10', 'capital_USD_p90'])
labor_sub_national_predictions = labor_sub_national_predictions.drop(columns=['jobs', 'jobs_p10', 'jobs_p90'])

capital_labor_predictions = capital_labor_predictions.merge(capital_sub_national_predictions, on='CAPITAL_PROJ_ID', how='left')
capital_labor_predictions = capital_labor_predictions.merge(labor_sub_national_predictions, on='LABOR_PROJ_ID', how='left')

# For pixels not in a sub-national region, scaling factor = 1
capital_labor_predictions[['capital_sub_scale', 'capital_sub_scale_p10', 'capital_sub_scale_p90', 'labor_sub_scale', 'labor_sub_scale_p10', 'labor_sub_scale_p90']] = (
    capital_labor_predictions[['capital_sub_scale', 'capital_sub_scale_p10', 'capital_sub_scale_p90', 'labor_sub_scale', 'labor_sub_scale_p10', 'labor_sub_scale_p90']].fillna(1)
)

### Step 3: Execute re-scaling
capital_labor_predictions['sub_scaled_capital_USD'] = capital_labor_predictions['capital_USD'] * capital_labor_predictions['capital_sub_scale']
capital_labor_predictions['sub_scaled_capital_USD_p10'] = capital_labor_predictions['capital_USD_p10'] * capital_labor_predictions['capital_sub_scale_p10']
capital_labor_predictions['sub_scaled_capital_USD_p90'] = capital_labor_predictions['capital_USD_p90'] * capital_labor_predictions['capital_sub_scale_p90']

capital_labor_predictions['sub_scaled_jobs'] = capital_labor_predictions['jobs'] * capital_labor_predictions['labor_sub_scale']
capital_labor_predictions['sub_scaled_jobs_p10'] = capital_labor_predictions['jobs_p10'] * capital_labor_predictions['labor_sub_scale_p10']
capital_labor_predictions['sub_scaled_jobs_p90'] = capital_labor_predictions['jobs_p90'] * capital_labor_predictions['labor_sub_scale_p90']

In [11]:
##### RE-SCALE TO NATIONAL (drop where known total doesn't exist)

### Step 1: Sum total predicted capital and labor in each country
# Add cross-walk of which country each pixel is in 
country_crosswalk = country_crosswalk.rename(columns={'GID_0': 'ISO3'})
capital_labor_predictions = capital_labor_predictions.merge(country_crosswalk, on=['x', 'y'], how='left')

# Get totals from initial predictions by country
capital_national_predictions = capital_labor_predictions[['ISO3', 'sub_scaled_capital_USD', 'sub_scaled_capital_USD_p10', 'sub_scaled_capital_USD_p90']].groupby('ISO3').sum().reset_index()
labor_national_predictions = capital_labor_predictions[['ISO3', 'sub_scaled_jobs', 'sub_scaled_jobs_p10', 'sub_scaled_jobs_p90']].groupby('ISO3').sum().reset_index()


### Step 2: Get re-scaling factor for each country
# Join predicted capital and labor to actual 
capital_national['national_total_capital_USD'] = capital_national['ag_capital_stock_mil_USD_nominal']  * 1e6
capital_national = capital_national.drop(columns=['ag_capital_stock_mil_USD_nominal'])

labor_national['national_total_jobs'] = labor_national['ag_labor_thousands_2020']  * 1e3
labor_national = labor_national.drop(columns=['ag_labor_thousands_2020'])

capital_national_predictions = capital_national_predictions.merge(capital_national[['ISO3', 'national_total_capital_USD']], on='ISO3', how='left')
labor_national_predictions = labor_national_predictions.merge(labor_national[['ISO3', 'national_total_jobs']], on='ISO3', how='left')

# Calculate rescaling factors 
capital_national_predictions['capital_nat_scale'] = capital_national_predictions['national_total_capital_USD'] / capital_national_predictions['sub_scaled_capital_USD'] 
capital_national_predictions['capital_nat_scale_p10'] = capital_national_predictions['national_total_capital_USD'] / capital_national_predictions['sub_scaled_capital_USD_p10'] 
capital_national_predictions['capital_nat_scale_p90'] = capital_national_predictions['national_total_capital_USD'] / capital_national_predictions['sub_scaled_capital_USD_p90'] 

labor_national_predictions['labor_nat_scale'] = labor_national_predictions['national_total_jobs'] / labor_national_predictions['sub_scaled_jobs'] 
labor_national_predictions['labor_nat_scale_p10'] = labor_national_predictions['national_total_jobs'] / labor_national_predictions['sub_scaled_jobs_p10'] 
labor_national_predictions['labor_nat_scale_p90'] = labor_national_predictions['national_total_jobs'] / labor_national_predictions['sub_scaled_jobs_p90'] 

# Merge back to data 
capital_national_predictions = capital_national_predictions.drop(columns=['sub_scaled_capital_USD', 'sub_scaled_capital_USD_p10', 'sub_scaled_capital_USD_p90'])
labor_national_predictions = labor_national_predictions.drop(columns=['sub_scaled_jobs', 'sub_scaled_jobs_p10', 'sub_scaled_jobs_p90'])

capital_labor_predictions = capital_labor_predictions.merge(capital_national_predictions, on='ISO3', how='left')
capital_labor_predictions = capital_labor_predictions.merge(labor_national_predictions, on='ISO3', how='left')


### Step 3: Execute re-scaling
capital_labor_predictions['rescaled_capital_USD'] = capital_labor_predictions['sub_scaled_capital_USD'] * capital_labor_predictions['capital_nat_scale']
capital_labor_predictions['rescaled_capital_USD_p10'] = capital_labor_predictions['sub_scaled_capital_USD_p10'] * capital_labor_predictions['capital_nat_scale_p10']
capital_labor_predictions['rescaled_capital_USD_p90'] = capital_labor_predictions['sub_scaled_capital_USD_p90'] * capital_labor_predictions['capital_nat_scale_p90']

capital_labor_predictions['rescaled_jobs'] = capital_labor_predictions['sub_scaled_jobs'] * capital_labor_predictions['labor_nat_scale']
capital_labor_predictions['rescaled_jobs_p10'] = capital_labor_predictions['sub_scaled_jobs_p10'] * capital_labor_predictions['labor_nat_scale_p10']
capital_labor_predictions['rescaled_jobs_p90'] = capital_labor_predictions['sub_scaled_jobs_p90'] * capital_labor_predictions['labor_nat_scale_p90']

In [18]:
##### SANITY CHECKS

# Check that total of known data matches total of raster data 
cap_err_per = (capital_labor_predictions['rescaled_capital_USD'].sum() / capital_labor_predictions['national_total_capital_USD'].sum()) * 100
lab_err_per = (capital_labor_predictions['rescaled_jobs'].sum() / capital_labor_predictions['national_total_jobs'].sum()) * 100

print("Percentage difference between sum of pixels and sum of countries:")
print(f"Capital: {cap_err_per:.2f}")
print(f"Labor: {lab_err_per:.2f}")

Percentage difference between sum of pixels and sum of countries:
Capital: 0.00
Labor: 0.00


In [19]:
##### SAVE FINAL OUTPUTS 

# save intermediate output
capital_labor_predictions.to_parquet(RESULTS_DIR / "intermediate_outputs/capital_labor_predictions_rescaled.parquet", index='False')

# clean for final output 
col_to_keep = ['x', 'y', 
               'rescaled_capital_USD', 'rescaled_capital_USD_p10', 'rescaled_capital_USD_p90',
               'rescaled_jobs', 'rescaled_jobs_p10', 'rescaled_jobs_p90'
]
capital_labor_final = capital_labor_predictions[col_to_keep]

capital_labor_final.to_parquet(RESULTS_DIR / "capital_labor_final.parquet", index='False')

# save rasters 

output_cols = [
    'rescaled_capital_USD',
    'rescaled_capital_USD_p10',
    'rescaled_capital_USD_p90',
    'rescaled_jobs',
    'rescaled_jobs_p10',
    'rescaled_jobs_p90'
]

for col in output_cols:
    out_path = RESULTS_DIR / f"{col}.tif" 
    save_column_as_raster(capital_labor_final, col, production_path, out_path)
    print(f"Saved {col} -> {out_path}")


ValueError: Index contains duplicate entries, cannot reshape